# Solar Filament Segmentation 2026 — Approach B: DINOv2 + Mask2Former-style masked-attention decoder

**Settings → Accelerator → GPU T4 x2, Internet ON**, then *Save & Run All* (about 4 h).

**Why this should beat the first U-Net.** The first U-Net (and Approach A) decide *which pixels belong together*
with hand-made rules: connected components, centre votes, merge heuristics. The biggest error bucket on this data is
exactly that: about a quarter of annotated filaments come out fragmented or merged. Here the model learns it:

* **Learned queries = filaments.** A Mask2Former-style decoder holds 80 learned queries. Each predicts one filament
  mask plus a confidence, and through **masked attention** it looks only inside its current mask while refining it,
  layer by layer, at strides 32 → 16 → 8. Because a query sees the whole disk, it can treat a filament broken by a
  faint gap as one object. Training uses Hungarian matching against the annotator's filaments.
* **DINOv2 ViT-B/14 context, partially fine-tuned.** The first 8 blocks are frozen; the last 4 adapt to H-alpha.
* **Detail where transformers are weak.** A CNN stem on the physics channels (strides 1–8) feeds the pixel decoder.
  A **full-resolution semantic head**, trained on the annotator-mean mask, gives the final boundaries: queries decide
  the grouping, the semantic map decides the edges.
* **Full frames at 1120 px** (no crops), so queries learn whole-disk filament counts.
* Trained with **DistributedDataParallel** on both T4s.
* **Three ways to form instances are compared on validation:** pure query masks (Mask2Former panoptic inference,
  edges trimmed by the semantic map); *hybrid* (connected components of the semantic map, joined when the same query
  covers them, which fixes fragmentation and falls back to plain components when queries are weak); and plain
  connected components. The best one makes the submission.

**Comparable numbers.** It trains on all images except the first U-Net notebook's validation images, and reports PQ
on exactly those images. Validation also tries plain connected components on the same model's semantic map, so you
can see what the learned grouping adds.

In [1]:
import os, sys, json, time, glob, subprocess, threading, importlib, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2, torch
from scipy import ndimage as ndi
warnings.filterwarnings("ignore")

DRY = os.environ.get("SFC_DRYRUN") == "1"          # tiny CPU smoke test only; leave unset on Kaggle
T0 = time.time()
WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(os.environ.get("SFC_WORK", "./work")).resolve()
WORK.mkdir(parents=True, exist_ok=True); os.chdir(WORK); sys.path.insert(0, str(WORK))
TEMP = Path("/kaggle/temp") if Path("/kaggle/temp").exists() else Path(os.environ.get("SFC_TEMP", "/tmp/sfc_temp"))
TEMP.mkdir(parents=True, exist_ok=True)
N_GPU = torch.cuda.device_count()

RES = int(os.environ.get("SFC_DRY_RES", 448)) if DRY else 1120     # multiple of 224 (DINOv2 patch 14 x FPN stride 32)
CFG = dict(
    backbone=os.environ.get("SFC_DRY_BACKBONE", "cnn") if DRY else "dinov2",   # "dinov2" or "cnn"
    vit="vit_small_patch14_reg4_dinov2" if DRY else "vit_base_patch14_reg4_dinov2.lvd142m",
    unfreeze=2 if DRY else 4,                     # last ViT blocks that are fine-tuned
    cnn="resnet18" if DRY else "resnet50.a1_in1k",   # used only if backbone == "cnn"
    d=128 if DRY else 256, mask_dim=64 if DRY else 128, queries=30 if DRY else 80, layers=3 if DRY else 6,
    batch=2,                                      # full frames per GPU
    lr=2e-4, lr_vit=1e-5, w_sem=2.0, seed=42,
    min_gt_area=max(int(12 * (RES / 1120) ** 2), 4),
    gt_dilate=max(2, round(4 * RES / 1120)),      # query "grouping masks" are trained on filaments dilated by this many px
    workers=0 if DRY else max((os.cpu_count() or 4) // max(N_GPU, 1), 1),
    log_every=10 if DRY else 100,
    max_steps=int(os.environ.get("SFC_DRY_STEPS", 30)) if DRY else 10 ** 9,
)
TRAIN_HOURS = 1.0 if DRY else 3.0
N_PROC = int(os.environ.get("SFC_DRY_NPROC", 1)) if DRY else max(N_GPU, 1)
print("GPUs:", N_GPU, [torch.cuda.get_device_name(g) for g in range(N_GPU)], "| training processes:", N_PROC)
if not DRY and N_GPU < 2:
    print("!!! fewer than 2 GPUs: training runs on one GPU (half the steps in the same time). Choose GPU T4 x2.")

GPUs: 2 ['Tesla T4', 'Tesla T4'] | training processes: 2


In [2]:
%%writefile sfc_common.py
"""sfc_common -- approach-independent utilities shared by both v3 notebooks (identical copy in each).

Data registry, physics-inspired input channels, memory-mapped caches, the competition metric (PQ), instance helpers,
the baseline validation split (for like-for-like comparison with the first U-Net submission) and submission export."""
from __future__ import annotations
import os, json, math, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, cv2
from scipy import ndimage as ndi
from pycocotools import mask as M

FULL = 2048
STATIONS = ("Bh", "Ch", "Lh", "Mh", "Th", "Uh")


# ============================================================================ data
def find_root():
    cands = [os.environ.get("SFC_ROOT"), "/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"]
    for c in cands:
        if c and (Path(c) / "train").exists():
            return Path(c)
    hits = sorted(Path("/kaggle/input").rglob("*Annotations*train*.json")) if Path("/kaggle/input").exists() else []
    assert hits, "competition data not found under /kaggle/input - attach the competition data to the notebook"
    return hits[0].parent.parent


class Data:
    """Annotation registry. `stems` has the same order as the first (baseline) notebook, so split indices match."""

    def __init__(self, root):
        root = Path(root)
        ann = sorted((root / "train").glob("*Annotations*train*.json"))[0]
        d = json.load(open(ann))
        self.anns, self.recs = {}, {}
        for a in d["annotations"]:
            self.anns.setdefault(a["image_id"], []).append(a)
        for im in d["images"]:
            self.recs.setdefault(im["file_name"].rsplit(".", 1)[0], []).append(im["id"])
        ext = (".jpeg", ".jpg", ".png")
        self.img_path = {p.stem: p for p in (root / "train" / "train_images").glob("*") if p.suffix.lower() in ext}
        self.stems = sorted(s for s in self.recs if s in self.img_path)
        self.test_paths = sorted(p for p in (root / "test" / "test_images").glob("*") if p.suffix.lower() in ext)

    def rles(self, rid, size):
        """One RLE per annotated filament of one annotator record (polygons are in 2048-px coordinates)."""
        s = size / FULL
        return [M.merge(M.frPyObjects([(np.asarray(p, float) * s).tolist() for p in a["segmentation"]], size, size))
                for a in self.anns.get(rid, [])]

    def labels(self, rid, size):
        lab = np.zeros((size, size), np.uint8)
        for k, r in enumerate(self.rles(rid, size), 1):
            lab[M.decode(r) > 0] = min(k, 255)
        return lab

    def gt(self, idx, size):
        return {r: self.rles(r, size) for i in idx for r in self.recs[self.stems[i]]}


def baseline_val_idx(n, seed=42, frac=0.15):
    """The exact held-out images of the first U-Net notebook (same seed / fraction / ordering)."""
    perm = np.random.default_rng(seed).permutation(n)
    return np.sort(perm[:max(int(round(frac * n)), 2)])


def make_folds(stems, k):
    """Observation-level folds, dealt round-robin after sorting by (station, timestamp): every fold sees every station
    and the whole time range; consecutive frames of the same filament fall in different folds."""
    order = sorted(range(len(stems)), key=lambda i: (stems[i][-2:], stems[i][:-2]))
    f = np.zeros(len(stems), int)
    for j, i in enumerate(order):
        f[i] = j % k
    return f


# ============================================================================ physics-inspired channels
def disk_fit(img8):
    b = cv2.GaussianBlur(img8, (0, 0), 3)
    _, m = cv2.threshold(b, 0, 1, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    h, w = img8.shape
    if n < 2:
        return w / 2, h / 2, 0.45 * min(h, w)
    comp = ndi.binary_fill_holes(lab == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA]))
    ys, xs = np.nonzero(comp)
    cx, cy, R = xs.mean(), ys.mean(), math.sqrt(comp.sum() / math.pi)
    if not (0.25 * w < R < 0.62 * w):                       # implausible fit: never let one odd frame break a run
        return w / 2, h / 2, 0.45 * min(h, w)
    return cx, cy, R


def preprocess(path, res):
    """-> chans (3,res,res) f16 [flat-field, local contrast, dark-ridge], gray (res,res) f16 (disk-standardised),
    valid (res,res) bool, disk (cx, cy, R).

    flat-field : intensity / radial median profile  (removes limb darkening, keeps local dark structures)
    contrast   : (flat - local mean) / local std    (normalised convolution restricted to the disk)
    dark ridge : max over scales of sigma^2 * largest Hessian eigenvalue (> 0 across dark, elongated structures)"""
    img8 = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    img8 = cv2.resize(img8, (res, res), interpolation=cv2.INTER_AREA)
    img = img8.astype(np.float32)
    cx, cy, R = disk_fit(img8)
    yy, xx = np.mgrid[0:res, 0:res].astype(np.float32)
    rr = np.hypot(xx - cx, yy - cy) / R
    valid = rr < 0.985
    nb = 100
    bins = np.clip((rr[valid] * nb).astype(int), 0, nb - 1)
    order = np.argsort(bins, kind="stable")
    b_s, v_s = bins[order], img[valid][order]
    bounds = np.searchsorted(b_s, np.arange(nb + 1))
    prof = np.array([np.median(v_s[bounds[i]:bounds[i + 1]]) if bounds[i + 1] - bounds[i] >= 20 else np.nan
                     for i in range(nb)], np.float32)
    cen = (np.arange(nb) + 0.5) / nb
    ok = ~np.isnan(prof)
    prof = np.interp(cen, cen[ok], prof[ok]) if ok.any() else np.full(nb, max(img[valid].mean(), 1.0))
    flat = np.where(valid, img / np.maximum(np.interp(rr, cen, prof), 1.0), 1.0).astype(np.float32)
    w = valid.astype(np.float32)
    s = res / 1024.0

    def nblur(a, sig):
        return cv2.GaussianBlur(a * w, (0, 0), sig) / np.maximum(cv2.GaussianBlur(w, (0, 0), sig), 1e-3)

    mu = nblur(flat, 16 * s)
    dlt = (flat - mu) * w
    z = dlt / np.maximum(np.sqrt(np.maximum(nblur(dlt ** 2, 32 * s), 1e-8)), 0.02)
    ridge = np.zeros_like(flat)
    for sig in (1.5 * s, 3 * s, 6 * s):
        g = cv2.GaussianBlur(flat, (0, 0), sig)
        gxx = cv2.Sobel(g, cv2.CV_32F, 2, 0, ksize=3); gyy = cv2.Sobel(g, cv2.CV_32F, 0, 2, ksize=3)
        gxy = cv2.Sobel(g, cv2.CV_32F, 1, 1, ksize=3)
        l1 = 0.5 * (gxx + gyy) + np.sqrt(0.25 * (gxx - gyy) ** 2 + gxy ** 2)
        ridge = np.maximum(ridge, sig ** 2 * np.maximum(l1, 0))
    ridge *= w
    ridge = np.log1p(ridge / (4 * np.median(ridge[valid]) + 1e-6)) if valid.any() else ridge
    chans = np.stack([np.clip((flat - 1) / 0.25, -3, 3) * w, np.clip(z, -4, 4) * w, np.clip(ridge, 0, 4) * w])
    gray = np.clip((img - img[valid].mean()) / (img[valid].std() + 1e-6), -5, 5) * w if valid.any() else img * 0
    return chans.astype(np.float16), gray.astype(np.float16), valid, (cx, cy, R)


# ============================================================================ memory-mapped caches
def build_cache(data, out_dir, res, test=False, n_jobs=None, log=print):
    """Train cache: chans, gray, valid, soft (annotator-mean mask, x255), labels (one uint8 label map per annotator
    record) + rec_start offsets. Test cache: chans, gray, valid. Resumes if a complete cache already exists."""
    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)
    man_p = out / "manifest.json"
    items = [p.stem for p in data.test_paths] if test else list(data.stems)
    if man_p.exists():
        man = json.load(open(man_p))
        if man.get("res") == res and man.get("items") == items:
            log(f"cache {out} already complete ({len(items)} images)")
            return man
    paths = data.test_paths if test else [data.img_path[s] for s in items]
    N = len(items)
    mm = lambda name, dt, shape: np.lib.format.open_memmap(out / f"{name}.npy", "w+", dt, shape)
    chans, gray, valid = mm("chans", np.float16, (N, 3, res, res)), mm("gray", np.float16, (N, res, res)), mm("valid", np.uint8, (N, res, res))
    disk = np.zeros((N, 3), np.float32)
    if not test:
        rec_start = np.cumsum([0] + [len(data.recs[s]) for s in items])
        soft, labels = mm("soft", np.uint8, (N, res, res)), mm("labels", np.uint8, (int(rec_start[-1]), res, res))

    def one(i):
        c, g, v, dk = preprocess(paths[i], res)
        chans[i], gray[i], valid[i], disk[i] = c, g, v, dk
        if not test:
            acc = np.zeros((res, res), np.float32)
            for j, rid in enumerate(data.recs[items[i]]):
                lab = data.labels(rid, res)
                labels[rec_start[i] + j] = lab
                acc += lab > 0
            soft[i] = np.round(acc / len(data.recs[items[i]]) * 255).astype(np.uint8)
        return i

    t = time.time()
    with ThreadPoolExecutor(n_jobs or os.cpu_count() or 1) as ex:
        for k, _ in enumerate(ex.map(one, range(N))):
            if k % 100 == 0 or k == N - 1:
                log(f"  cached {k + 1}/{N} ({time.time() - t:.0f}s)")
    for a in (chans, gray, valid) + (() if test else (soft, labels)):
        a.flush()
    np.save(out / "disk.npy", disk)
    if not test:
        np.save(out / "rec_start.npy", rec_start)
    man = {"res": res, "items": items, "test": test}
    json.dump(man, open(man_p, "w"))
    return man


def open_cache(out_dir):
    out = Path(out_dir)
    c = {"manifest": json.load(open(out / "manifest.json"))}
    for name in ("chans", "gray", "valid", "soft", "labels"):
        if (out / f"{name}.npy").exists():
            c[name] = np.load(out / f"{name}.npy", mmap_mode="r")
    c["disk"] = np.load(out / "disk.npy")
    if (out / "rec_start.npy").exists():
        c["rec_start"] = np.load(out / "rec_start.npy")
    return c


# ============================================================================ metric
def pq(gt, preds):
    """Competition metric: every annotator record is scored separately against the image's predictions, then pooled.
    hit = IoU > 0.5 for each (gt, prediction) cell; PQ = sum IoU(hits) / (TP + FP/2 + FN/2)."""
    S = TP = FP = FN = 0.0
    for rid, gts in gt.items():
        pr = preds.get(rid.split("-", 1)[1], [])
        if not gts:
            FP += len(pr); continue
        if not pr:
            FN += len(gts); continue
        iou = np.asarray(M.iou(pr, gts, [0] * len(gts))).T
        hit = iou > 0.5
        S += iou[hit].sum(); TP += hit.sum(); FP += (hit.sum(0) == 0).sum(); FN += (hit.sum(1) == 0).sum()
    den = TP + 0.5 * FP + 0.5 * FN
    return dict(pq=float(S / den) if den else 0.0, sq=float(S / TP) if TP else 0.0, dq=float(TP / den) if den else 0.0,
                tp=int(TP), fp=int(FP), fn=int(FN))


def matched_iou(lab, gt_rles_by_rec):
    """Per component of `lab`: mean over annotator records of IoU * 1[IoU > 0.5] (its expected contribution to PQ's
    numerator). With disjoint predictions at most one GT per record can exceed 0.5."""
    n = int(lab.max())
    if n == 0:
        return np.zeros(0)
    crl = [M.encode(np.asfortranarray((lab == k).astype(np.uint8))) for k in range(1, n + 1)]
    acc = np.zeros(n)
    for rid, gts in gt_rles_by_rec.items():
        if gts:
            iou = np.asarray(M.iou(crl, gts, [0] * len(gts)))       # (n, G)
            acc += (iou * (iou > 0.5)).sum(1)
    return acc / max(len(gt_rles_by_rec), 1)


# ============================================================================ instance helpers
def instances_cc(prob, valid, t_hi, t_lo, min_area, close=1):
    """Hysteresis threshold -> small closing -> connected components (disjoint by construction)."""
    ext = ((prob >= t_lo) & valid).astype(np.uint8)
    n, lab = cv2.connectedComponents(ext, connectivity=8)
    if n <= 1:
        return np.zeros(prob.shape, np.int32)
    seeds = np.unique(lab[(prob >= t_hi) & valid]); seeds = seeds[seeds > 0]
    m = np.isin(lab, seeds)
    if close > 0 and m.any():
        k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * close + 1, 2 * close + 1))
        m = cv2.morphologyEx(m.astype(np.uint8), cv2.MORPH_CLOSE, k).astype(bool) & valid
    n, lab, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8), connectivity=8)
    keep = np.zeros(n, np.int32)
    ids = np.flatnonzero(st[:, cv2.CC_STAT_AREA] >= min_area); ids = ids[ids > 0]
    keep[ids] = np.arange(1, len(ids) + 1)
    return keep[lab]


def relabel(lab):
    u, inv = np.unique(lab, return_inverse=True)
    out = inv.reshape(lab.shape).astype(np.int32)
    return out if u[0] == 0 else out + 1


def to_rles(lab, keep, prob=None, thr=None, size=FULL):
    """RLEs of the kept labels at `size`: nearest-neighbour label up-sampling, trimmed by the bilinearly up-sampled
    probability so boundaries follow the map rather than pixel blocks. Disjoint by construction."""
    if size != lab.shape[0]:
        ix = (np.arange(size) * lab.shape[0]) // size
        lab = lab[ix][:, ix]
        if prob is not None and thr:
            lab = lab * (cv2.resize(prob.astype(np.float32), (size, size), interpolation=cv2.INTER_LINEAR) >= thr)
    out = []
    for k in keep:
        m = lab == k
        if m.any():
            out.append(M.encode(np.asfortranarray(m.astype(np.uint8))))
    return out


def label_rles(lab):
    return {int(k): M.encode(np.asfortranarray((lab == k).astype(np.uint8))) for k in np.unique(lab) if k > 0}


def select(score, thr, min_keep=1):
    """Labels (1-based) with score >= thr; never fewer than min_keep (every training record has >= 1 filament)."""
    score = np.asarray(score)
    keep = np.flatnonzero(score >= thr) + 1
    if len(keep) < min_keep and len(score):
        keep = np.argsort(-score)[:min_keep] + 1
    return keep.tolist()


def rot_flip(a, k, f):
    a = np.rot90(a, k, axes=(-2, -1))
    return np.ascontiguousarray(np.flip(a, -1) if f else a)


# ============================================================================ submission
def write_submission(preds, path):
    rows = [{"filament_id": f"{s}_{k}", "segmentation_rle": r["counts"].decode("ascii") if isinstance(r["counts"], bytes) else r["counts"]}
            for s in sorted(preds) for k, r in enumerate(preds[s], 1)]
    df = pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"])
    df.to_csv(path, index=False)
    return df


def validate_submission(df, test_stems):
    img = df.filament_id.str.rsplit("_", n=1).str[0]
    assert df.filament_id.is_unique, "duplicate filament_id"
    unknown = set(img) - set(test_stems)
    assert not unknown, f"rows for unknown images {sorted(unknown)[:3]}"
    for s, g in df.groupby(img):
        L = M.decode([{"size": [FULL, FULL], "counts": c.encode()} for c in g.segmentation_rle])
        assert L.reshape(-1, L.shape[2]).sum(0).min() > 0, f"{s}: empty mask"
        assert L.sum(2).max() <= 1, f"{s}: overlapping masks"
    missing = sorted(set(test_stems) - set(img))
    return {"rows": len(df), "images": int(img.nunique()), "per_image": round(len(df) / max(len(test_stems), 1), 2),
            "images_without_rows": missing}

Writing sfc_common.py


In [3]:
%%writefile b_lib.py
"""b_lib -- Approach B: DINOv2 (last blocks fine-tuned) + CNN detail stem + FPN pixel decoder + Mask2Former-style
masked-attention transformer decoder (learned queries = filaments), Hungarian matching, and a full-resolution semantic
head whose map gives the final mask boundaries.

Training runs as `torchrun --nproc_per_node=<gpus> b_lib.py train ...` (DistributedDataParallel, one process per GPU)."""
from __future__ import annotations
import os, sys, json, math, time, argparse
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import torch.distributed as dist
from scipy.optimize import linear_sum_assignment
import sfc_common as S


# ============================================================================ building blocks
def cbr(cin, cout, stride=1):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, stride, 1, bias=False), nn.GroupNorm(8 if cout % 8 == 0 else 1, cout), nn.GELU())


def up(x, ref):
    return F.interpolate(x, size=ref.shape[-2:], mode="bilinear", align_corners=False)


def sine_pos(h, w, d, dev):
    """DETR-style 2-D sine positional encoding, (d, h, w)."""
    npf = d // 2
    y = (torch.arange(h, device=dev, dtype=torch.float32) + 0.5) / h * 2 * math.pi
    x = (torch.arange(w, device=dev, dtype=torch.float32) + 0.5) / w * 2 * math.pi
    dim_t = 10000 ** (2 * (torch.arange(npf, device=dev) // 2).float() / npf)
    py, px = y[:, None] / dim_t, x[:, None] / dim_t
    py = torch.stack([py[:, 0::2].sin(), py[:, 1::2].cos()], -1).flatten(1)
    px = torch.stack([px[:, 0::2].sin(), px[:, 1::2].cos()], -1).flatten(1)
    return torch.cat([py[:, None, :].expand(h, w, npf), px[None, :, :].expand(h, w, npf)], -1).permute(2, 0, 1)


class MLP(nn.Module):
    def __init__(self, d_in, d_hid, d_out, n):
        super().__init__()
        dims = [d_in] + [d_hid] * (n - 1) + [d_out]
        self.layers = nn.ModuleList(nn.Linear(a, b) for a, b in zip(dims[:-1], dims[1:]))

    def forward(self, x):
        for i, l in enumerate(self.layers):
            x = l(x) if i == len(self.layers) - 1 else F.relu(l(x))
        return x


# ============================================================================ backbones (context features at strides 8 / 16 / 32)
class DinoContext(nn.Module):
    """DINOv2 with registers. The first blocks run frozen (no gradients); the last `unfreeze` blocks are fine-tuned.
    Features from three depths are concatenated and resampled to strides 8, 16 and 32."""

    def __init__(self, name, unfreeze, d=256):
        super().__init__()
        import timm
        self.vit = timm.create_model(name, pretrained=False, dynamic_img_size=True, num_classes=0)
        nb = len(self.vit.blocks)
        self.n_frozen = nb - unfreeze
        self.taps = [nb // 3 - 1, 2 * nb // 3 - 1, nb - 1]
        for i, p in self.vit.named_parameters():
            p.requires_grad = i.startswith("blocks.") and int(i.split(".")[1]) >= self.n_frozen or (unfreeze > 0 and i.startswith("norm."))
        D = self.vit.embed_dim
        self.proj = nn.ModuleList(nn.Sequential(nn.Conv2d(3 * D, d, 1), nn.GroupNorm(32, d)) for _ in range(3))
        self.out_ch = (d, d, d)

    def forward(self, gray):
        B, _, H, W = gray.shape
        x = F.pad(gray, (0, (-W) % 14, 0, (-H) % 14)).expand(-1, 3, -1, -1)
        v = self.vit
        h, w = x.shape[-2] // 14, x.shape[-1] // 14
        with torch.set_grad_enabled(torch.is_grad_enabled() and self.n_frozen == 0):
            t = v.norm_pre(v.patch_drop(v._pos_embed(v.patch_embed(x))))
        feats = []
        for i, blk in enumerate(v.blocks):
            with torch.set_grad_enabled(torch.is_grad_enabled() and i >= self.n_frozen):
                t = blk(t)
            if i in self.taps:
                feats.append(v.norm(t) if i == len(v.blocks) - 1 else t)
        npt = v.num_prefix_tokens
        f = torch.cat([z[:, npt:].transpose(1, 2).reshape(B, -1, h, w) for z in feats], 1)
        outs = []
        for proj, s in zip(self.proj, (8, 16, 32)):
            hs, ws = math.ceil(H / s), math.ceil(W / s)
            # resample the 14-px grid onto stride s (crop away the bottom/right padding first)
            fc = f[..., :math.ceil(H / 14), :math.ceil(W / 14)]
            outs.append(proj(F.interpolate(fc, size=(hs, ws), mode="bilinear", align_corners=False)))
        return outs


class CNNContext(nn.Module):
    def __init__(self, name):
        super().__init__()
        import timm
        self.enc = timm.create_model(name, pretrained=False, features_only=True, in_chans=3, out_indices=(2, 3, 4))
        self.out_ch = tuple(self.enc.feature_info.channels())

    def forward(self, phys):
        return self.enc(phys)


# ============================================================================ the model
class FilamentMask2Former(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        d, md, Q = cfg["d"], cfg["mask_dim"], cfg["queries"]
        self.kind = cfg["backbone"]
        self.ctx = DinoContext(cfg["vit"], cfg["unfreeze"], d) if self.kind == "dinov2" else CNNContext(cfg["cnn"])
        # CNN detail stem on the physics channels: strides 1, 2, 4, 8
        self.s1 = nn.Sequential(cbr(3, 32), cbr(32, 32))
        self.s2 = nn.Sequential(cbr(32, 64, 2), cbr(64, 64))
        self.s4 = nn.Sequential(cbr(64, 128, 2), cbr(128, 128))
        self.s8 = nn.Sequential(cbr(128, 192, 2), cbr(192, 192))
        c8, c16, c32 = self.ctx.out_ch
        # FPN pixel decoder
        self.l32, self.l16, self.l8 = nn.Conv2d(c32, d, 1), nn.Conv2d(c16, d, 1), nn.Conv2d(c8, d, 1)
        self.l8s, self.l4s = nn.Conv2d(192, d, 1), nn.Conv2d(128, d, 1)
        self.o32, self.o16, self.o8, self.o4 = cbr(d, d), cbr(d, d), cbr(d, d), cbr(d, d)
        self.mask_proj = nn.Conv2d(d, md, 1)
        # full-resolution semantic head (annotator-mean target) -> final boundaries
        self.sem2 = cbr(d + 64, 64)
        self.sem1 = nn.Sequential(cbr(64 + 32, 32), nn.Conv2d(32, 1, 1))
        # masked-attention transformer decoder
        self.Q, self.L, self.nh = Q, cfg["layers"], 8
        self.query_feat, self.query_pos, self.level = nn.Embedding(Q, d), nn.Embedding(Q, d), nn.Embedding(3, d)
        self.cross = nn.ModuleList(nn.MultiheadAttention(d, self.nh, batch_first=True) for _ in range(self.L))
        self.selfa = nn.ModuleList(nn.MultiheadAttention(d, self.nh, batch_first=True) for _ in range(self.L))
        self.ffn = nn.ModuleList(nn.Sequential(nn.Linear(d, 4 * d), nn.ReLU(), nn.Linear(4 * d, d)) for _ in range(self.L))
        self.norms = nn.ModuleList(nn.ModuleList(nn.LayerNorm(d) for _ in range(3)) for _ in range(self.L))
        self.dec_norm = nn.LayerNorm(d)
        self.cls = nn.Linear(d, 2)                               # [filament, no-object]
        self.mask_embed = MLP(d, d, md, 3)

    def _pred(self, q, mfeat, attn_hw):
        qn = self.dec_norm(q)
        cls = self.cls(qn)
        mlog = torch.einsum("bqc,bchw->bqhw", self.mask_embed(qn), mfeat)
        with torch.no_grad():
            am = F.interpolate(mlog.float(), size=attn_hw, mode="bilinear", align_corners=False).sigmoid().flatten(2) < 0.5
            am[am.all(-1)] = False                               # a query predicting nothing may attend everywhere
            am = am.repeat_interleave(self.nh, 0)
        return cls, mlog, am

    def forward(self, phys, gray):
        s1 = self.s1(phys); s2 = self.s2(s1); s4 = self.s4(s2); s8 = self.s8(s4)
        c8, c16, c32 = self.ctx(gray) if self.kind == "dinov2" else self.ctx(phys)
        p32 = self.o32(self.l32(c32))
        p16 = self.o16(self.l16(c16) + up(p32, c16))
        p8 = self.o8(self.l8(c8) + self.l8s(s8) + up(p16, s8))
        p4 = self.o4(self.l4s(s4) + up(p8, s4))
        mfeat = self.mask_proj(p4)
        sem = self.sem1(torch.cat([up(self.sem2(torch.cat([up(p4, s2), s2], 1)), s1), s1], 1))
        # decoder: round-robin over strides 32 -> 16 -> 8
        mems = [p32, p16, p8]
        B = phys.shape[0]
        srcs = [(m.flatten(2).transpose(1, 2) + self.level.weight[l]) for l, m in enumerate(mems)]
        poss = [sine_pos(m.shape[-2], m.shape[-1], m.shape[1], m.device).flatten(1).t()[None].to(m.dtype) for m in mems]
        q = self.query_feat.weight[None].expand(B, -1, -1)
        qp = self.query_pos.weight[None].expand(B, -1, -1)
        cls, mlog, am = self._pred(q, mfeat, mems[0].shape[-2:])
        outs = [(cls, mlog)]
        for i in range(self.L):
            l = i % 3
            n1, n2, n3 = self.norms[i]
            q = n1(q + self.cross[i](q + qp, srcs[l] + poss[l], srcs[l], attn_mask=am, need_weights=False)[0])
            q = n2(q + self.selfa[i](q + qp, q + qp, q, need_weights=False)[0])
            q = n3(q + self.ffn[i](q))
            cls, mlog, am = self._pred(q, mfeat, mems[(i + 1) % 3].shape[-2:])
            outs.append((cls, mlog))
        return outs, sem


def build_model(cfg, ctx_init=None):
    m = FilamentMask2Former(cfg)
    if ctx_init and os.path.exists(ctx_init):
        sd = torch.load(ctx_init, map_location="cpu", weights_only=True)
        (m.ctx.vit if m.kind == "dinov2" else m.ctx.enc).load_state_dict(sd, strict=m.kind != "dinov2")
    return m


def save_ctx_init(cfg, out_path, log=print):
    """Parent process: fetch the pretrained backbone once (hub, else an attached file) for the training processes."""
    import timm, glob
    if cfg["backbone"] == "dinov2":
        name, kw, key = cfg["vit"], dict(dynamic_img_size=True, num_classes=0), "dinov2"
    else:
        name, kw, key = cfg["cnn"], dict(features_only=True, in_chans=3, out_indices=(2, 3, 4)), cfg["cnn"].split(".")[0]
    try:
        torch.save(timm.create_model(name, pretrained=True, **kw).state_dict(), out_path); return "timm hub"
    except Exception as e:
        log(f"  hub download failed ({type(e).__name__})")
    for p in sorted(glob.glob("/kaggle/input/**/*", recursive=True)):
        if key in p.lower() and p.endswith((".safetensors", ".pth", ".bin")):
            try:
                torch.save(timm.create_model(name, pretrained=True, pretrained_cfg_overlay=dict(file=p), **kw).state_dict(), out_path)
                return p
            except Exception:
                pass
    return None


# ============================================================================ matching + losses
def targets_from_labels(lab, min_area, dilate=0):
    """(R,R) uint8 label map of one annotator record -> (G, R/4, R/4) soft "grouping" masks.
    Filaments are only a few pixels wide, i.e. sub-cell at the queries' stride 4, so each instance is first dilated by
    `dilate` px: the queries learn WHICH pixels belong together, the full-resolution semantic head draws the edges."""
    ids, cnt = torch.unique(lab, return_counts=True)
    ids = ids[(ids > 0) & (cnt >= min_area)]
    if len(ids) == 0:
        return lab.new_zeros((0, lab.shape[0] // 4, lab.shape[1] // 4), dtype=torch.float32)
    m = (lab[None] == ids[:, None, None]).float()[:, None]
    if dilate > 0:
        m = F.max_pool2d(m, 2 * dilate + 1, 1, dilate)
    return F.avg_pool2d(m, 4)[:, 0]


@torch.no_grad()
def match(cls, mlog, tgt, n_points=12544):
    G = tgt.shape[0]
    if G == 0:
        return np.zeros(0, int), np.zeros(0, int)
    prob = cls.float().softmax(-1)[:, 0]
    m, t = mlog.float().flatten(1), tgt.flatten(1)
    idx = torch.randint(0, m.shape[1], (min(n_points, m.shape[1]),), device=m.device)
    m, t = m[:, idx], t[:, idx]
    pos = F.binary_cross_entropy_with_logits(m, torch.ones_like(m), reduction="none")
    neg = F.binary_cross_entropy_with_logits(m, torch.zeros_like(m), reduction="none")
    c_bce = (pos @ t.t() + neg @ (1 - t).t()) / m.shape[1]
    s = m.sigmoid()
    c_dice = 1 - (2 * s @ t.t() + 1) / (s.sum(-1)[:, None] + t.sum(-1)[None] + 1)
    C = -2 * prob[:, None] + 5 * c_bce + 5 * c_dice
    return linear_sum_assignment(C.cpu().numpy())


def set_loss(outs, tgts, valid4, w_noobj=0.1):
    """Sum over decoder layers of: class CE (all queries) + mask BCE + mask Dice (matched queries only)."""
    total, parts = 0.0, np.zeros(3)
    n_masks = max(sum(t.shape[0] for t in tgts), 1)
    for cls, mlog in outs:
        lc = lm = ld = 0.0
        for b, tgt in enumerate(tgts):
            qi, gi = match(cls[b], mlog[b], tgt)
            label = torch.ones(cls.shape[1], dtype=torch.long, device=cls.device)       # 1 = no-object
            label[torch.as_tensor(qi, device=cls.device, dtype=torch.long)] = 0
            lc = lc + F.cross_entropy(cls[b].float(), label, weight=torch.tensor([1.0, w_noobj], device=cls.device))
            if len(qi):
                m = mlog[b, torch.as_tensor(qi, device=cls.device)].float()
                t = tgt[torch.as_tensor(gi, device=cls.device)]
                v = valid4[b]
                lm = lm + ((F.binary_cross_entropy_with_logits(m, t, reduction="none") * v).flatten(1).sum(1) / v.sum().clamp_min(1)).sum()
                s = m.sigmoid() * v
                ld = ld + (1 - (2 * (s * t).flatten(1).sum(1) + 1) / (s.flatten(1).sum(1) + (t * v).flatten(1).sum(1) + 1)).sum()
        lc, lm, ld = lc / len(tgts), lm / n_masks, ld / n_masks
        total = total + 2 * lc + 5 * lm + 5 * ld
        parts += np.array([float(v.detach()) if torch.is_tensor(v) else float(v) for v in (lc, lm, ld)])
    return total, parts / len(outs)


def sem_loss(logit, soft, valid):
    bce = (F.binary_cross_entropy_with_logits(logit.float(), soft, reduction="none") * valid).sum() / valid.sum().clamp_min(1)
    p = torch.sigmoid(logit.float()) * valid
    return bce + 1 - (2 * (p * soft).sum() + 1) / (p.sum() + (soft * valid).sum() + 1)


# ============================================================================ data
class FrameDS(torch.utils.data.Dataset):
    """Full frames (no crops: the queries must learn whole-disk filament counts). One annotator record per sample
    supplies the instance targets; the semantic target is always the annotator mean. D4 + photometric augmentation."""

    def __init__(self, cache_dir, idx, length):
        self.cache_dir, self.idx, self.length, self.c = cache_dir, np.asarray(idx), length, None

    def __len__(self):
        return self.length

    def __getitem__(self, k):
        if self.c is None:
            self.c = S.open_cache(self.cache_dir)
        c, r = self.c, np.random
        i = int(r.choice(self.idx))
        j = int(c["rec_start"][i] + r.randint(c["rec_start"][i + 1] - c["rec_start"][i]))
        kk, ff = r.randint(4), r.rand() < 0.5
        phys = S.rot_flip(np.asarray(c["chans"][i], np.float32), kk, ff)
        gray = S.rot_flip(np.asarray(c["gray"][i], np.float32), kk, ff)
        soft = S.rot_flip(np.asarray(c["soft"][i], np.float32) / 255, kk, ff)
        valid = S.rot_flip(np.asarray(c["valid"][i], np.float32), kk, ff)
        lab = S.rot_flip(np.asarray(c["labels"][j]), kk, ff)
        phys = (phys * r.uniform(0.85, 1.15, (3, 1, 1)) + r.normal(0, 0.05, (3, 1, 1))) * valid
        gray = (gray * r.uniform(0.85, 1.15) + r.normal(0, 0.05)) * valid
        t = lambda a: torch.from_numpy(np.ascontiguousarray(a))
        return (t(phys.astype(np.float32)), t(gray[None].astype(np.float32)), t(soft[None].astype(np.float32)),
                t(valid[None].astype(np.float32)), t(lab))


# ============================================================================ training (DDP)
def train_main(a):
    cfg = json.load(open(os.path.join(a.work, "cfg_b.json")))
    ddp = "LOCAL_RANK" in os.environ
    cuda = torch.cuda.is_available()
    if ddp:
        dist.init_process_group("nccl" if cuda else "gloo")
        rank, world, local = dist.get_rank(), dist.get_world_size(), int(os.environ["LOCAL_RANK"])
    else:
        rank, world, local = 0, 1, 0
    if cuda:
        torch.cuda.set_device(local)
    dev = torch.device(f"cuda:{local}" if cuda else "cpu")
    torch.manual_seed(cfg["seed"] + rank); np.random.seed(cfg["seed"] + rank)
    torch.backends.cudnn.benchmark = True
    tr = np.array(json.load(open(os.path.join(a.work, "train_idx_b.json"))))
    model = build_model(cfg, os.path.join(a.work, "ctx_init_b.pt")).to(dev)
    vit_p = [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("ctx.vit.")]
    no_wd = lambda n, p: p.ndim == 1 or "embed" in n or "level" in n or "query" in n
    other = [(n, p) for n, p in model.named_parameters() if p.requires_grad and not n.startswith("ctx.vit.")]
    groups = [{"params": [p for n, p in other if not no_wd(n, p)], "base": cfg["lr"], "weight_decay": 0.05},
              {"params": [p for n, p in other if no_wd(n, p)], "base": cfg["lr"], "weight_decay": 0.0}]
    if vit_p:
        groups.append({"params": vit_p, "base": cfg["lr_vit"], "weight_decay": 0.05})
    opt = torch.optim.AdamW(groups)
    if ddp:
        model = nn.parallel.DistributedDataParallel(model, device_ids=[local] if cuda else None)
    net = model.module if ddp else model
    amp = cuda
    scaler = torch.amp.GradScaler("cuda", enabled=amp)
    dl = torch.utils.data.DataLoader(FrameDS(a.cache, tr, 10 ** 8), batch_size=cfg["batch"], num_workers=cfg["workers"],
                                     drop_last=True, persistent_workers=cfg["workers"] > 0, pin_memory=cuda,
                                     worker_init_fn=lambda w: np.random.seed((cfg["seed"] * 7919 + rank * 101 + w + int(time.time())) % 2 ** 31))
    budget, t0, step, hist = a.hours * 3600, time.time(), 0, []
    warm = min(500, max(cfg["max_steps"] // 10, 20))
    min_area = cfg["min_gt_area"]
    if rank == 0:
        n_tr = sum(p.numel() for p in net.parameters() if p.requires_grad) / 1e6
        print(f"{world} process(es) | {len(tr)} train frames | batch {cfg['batch']}/process | {n_tr:.1f} M trainable | {a.hours:.2f} h", flush=True)
    model.train()
    for phys, gray, soft, valid, lab in dl:
        stop = torch.tensor([float((time.time() - t0) >= budget or step >= cfg["max_steps"])], device=dev)
        if ddp:
            dist.all_reduce(stop, op=dist.ReduceOp.MAX)          # every rank stops at the same step (no DDP hang)
        if stop.item() > 0:
            break
        frac = min((time.time() - t0) / budget, 1.0)
        for g in opt.param_groups:
            g["lr"] = g["base"] * min(1.0, (step + 1) / warm) * (0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * frac)))
        phys, gray, soft, valid, lab = (t.to(dev, non_blocking=True) for t in (phys, gray, soft, valid, lab))
        with torch.autocast(device_type=dev.type, enabled=amp):
            outs, sem = model(phys, gray)
        tgts = [targets_from_labels(lab[b], min_area, cfg["gt_dilate"]) for b in range(lab.shape[0])]
        valid4 = F.avg_pool2d(valid, 4)[:, 0]
        l_set, parts = set_loss(outs, tgts, valid4)
        l_sem = sem_loss(sem, soft, valid)
        loss = l_set + cfg["w_sem"] * l_sem
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        hist.append([loss.item(), *parts, l_sem.item()]); step += 1
        if rank == 0 and step % cfg["log_every"] == 0:
            m = np.mean(hist[-cfg["log_every"]:], 0); el = time.time() - t0
            print(f"step {step} | {el / 60:.0f} min | {step * cfg['batch'] * world / el:.2f} frames/s | loss {m[0]:.3f} "
                  f"(cls {m[1]:.3f} mask-bce {m[2]:.3f} dice {m[3]:.3f} sem {m[4]:.3f})", flush=True)
    if rank == 0:
        torch.save(net.state_dict(), os.path.join(a.work, "b_model.pt"))
        json.dump({"steps": step, "minutes": (time.time() - t0) / 60, "processes": world,
                   "final_loss": float(np.mean([h[0] for h in hist[-100:]])) if hist else None},
                  open(os.path.join(a.work, "b_train.json"), "w"))
        print(f"done: {step} steps in {(time.time() - t0) / 60:.0f} min", flush=True)
    if ddp:
        dist.barrier(); dist.destroy_process_group()


# ============================================================================ inference + panoptic assembly
@torch.no_grad()
def infer(model, phys, gray, valid, dev):
    """-> filament probability per query (Q,), query mask logits at stride 4 (Q, R/4, R/4) float16,
    semantic probability (R, R) averaged over an identity and a horizontally flipped pass."""
    v = torch.from_numpy(np.asarray(valid, np.float32))[None, None].to(dev)
    p = torch.from_numpy(np.asarray(phys, np.float32))[None].to(dev) * v
    g = torch.from_numpy(np.asarray(gray, np.float32))[None, None].to(dev) * v
    with torch.autocast(device_type=dev.type, enabled=dev.type == "cuda"):
        outs, sem = model(p, g)
        _, sem2 = model(torch.flip(p, (-1,)), torch.flip(g, (-1,)))
    cls, mlog = outs[-1]
    semp = 0.5 * (torch.sigmoid(sem.float()) + torch.flip(torch.sigmoid(sem2.float()), (-1,)))
    return (cls[0].float().softmax(-1)[:, 0].cpu().numpy(), mlog[0].half().cpu().numpy(),
            (semp[0, 0] * v[0, 0]).cpu().numpy())


def panoptic(cls_p, mlog, sem, valid, obj_thr, t_sem, min_area, overlap=0.8):
    """Mask2Former panoptic inference: each pixel goes to the query with the highest score x mask probability (if that
    probability > 0.5); a query survives only if it keeps >= `overlap` of its own mask. Optionally the full-resolution
    semantic map (>= t_sem) trims the boundaries. Returns (label image, score per label)."""
    R = sem.shape[0]
    keep = np.flatnonzero(cls_p >= obj_thr)
    if len(keep) == 0:
        return np.zeros((R, R), np.int32), np.zeros(0)
    m = torch.sigmoid(F.interpolate(torch.from_numpy(mlog[keep]).float()[None], size=(R, R), mode="bilinear", align_corners=False)[0])
    m = m * torch.from_numpy(np.asarray(valid, np.float32))
    b = m > 0.5
    area0 = b.flatten(1).sum(1).float()
    score = torch.from_numpy(cls_p[keep]).float() * ((m * b).flatten(1).sum(1) / area0.clamp_min(1))
    best = (score[:, None, None] * m).argmax(0)
    lab = torch.where(m.gather(0, best[None])[0] > 0.5, best + 1, torch.zeros_like(best))
    # overlap test on the query masks themselves (before trimming: grouping masks are deliberately wider than filaments)
    cnt0 = torch.bincount(lab.flatten(), minlength=len(keep) + 1)[1:].float()
    if t_sem > 0:
        lab = lab * torch.from_numpy(sem >= t_sem)
    cnt = torch.bincount(lab.flatten(), minlength=len(keep) + 1)[1:].float()
    ok = (cnt0 >= overlap * area0) & (cnt >= min_area) & (area0 > 0)
    remap = torch.zeros(len(keep) + 1, dtype=torch.long)
    remap[1:][ok] = torch.arange(1, int(ok.sum()) + 1)
    return remap[lab].numpy().astype(np.int32), score[ok].numpy()


def hybrid(cls_p, mlog, sem, valid, obj_thr, t_lo, t_hi, min_area, min_cover=0.5):
    """Connected components of the semantic map form the instances; the learned queries only VOTE on grouping:
    components mostly covered by the same confident query are joined into one filament (fixes fragmentation).
    With weak queries this degrades gracefully to plain connected components. Returns (label image, mean-prob score)."""
    from scipy import ndimage as ndi
    lab = S.instances_cc(sem, valid, t_hi, t_lo, min_area)
    n = int(lab.max())
    keep = np.flatnonzero(cls_p >= obj_thr)
    if n > 1 and len(keep):
        R = sem.shape[0]
        m = torch.sigmoid(F.interpolate(torch.from_numpy(mlog[keep]).float()[None], size=(R, R), mode="bilinear", align_corners=False)[0]) > 0.5
        labt = torch.from_numpy(lab).long()
        area = torch.bincount(labt.flatten(), minlength=n + 1)[1:].float().clamp_min(1)
        cover = torch.stack([torch.bincount(labt[m[q]], minlength=n + 1)[1:].float() / area for q in range(len(keep))])   # (K, n)
        best_cov, best_q = cover.max(0)
        group = torch.where(best_cov >= min_cover, best_q, -(torch.arange(n) + 1)).numpy()
        _, root = np.unique(group, return_inverse=True)
        lab = np.concatenate([[0], root + 1])[lab].astype(np.int32)
    n = int(lab.max())
    return lab, (np.asarray(ndi.mean(sem, lab, np.arange(1, n + 1))) if n else np.zeros(0))


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("cmd"); ap.add_argument("--hours", type=float, required=True)
    ap.add_argument("--work", required=True); ap.add_argument("--cache", required=True)
    a = ap.parse_args()
    try:
        train_main(a)
    except BaseException:
        import traceback; traceback.print_exc(); sys.exit(1)

Writing b_lib.py


In [4]:
importlib.invalidate_caches()
import sfc_common as S, b_lib as B
importlib.reload(S); importlib.reload(B)


def banner(t):
    print(f"\n{'=' * 90}\n[{(time.time() - T0) / 3600:.2f} h] {t}\n{'=' * 90}", flush=True)

## 1 · Data, physics channels and caches

In [5]:
banner("DATA + CACHE")
data = S.Data(S.find_root())
stems = data.stems
print(f"{len(stems)} train images, {sum(len(data.recs[s]) for s in stems)} annotator records, {len(data.test_paths)} test images")
S.build_cache(data, TEMP / f"train_{RES}", RES)
S.build_cache(data, TEMP / f"test_{RES}", RES, test=True)
C, TC = S.open_cache(TEMP / f"train_{RES}"), S.open_cache(TEMP / f"test_{RES}")
VAL = S.baseline_val_idx(len(stems))                       # the first U-Net notebook's validation images
TR = np.setdiff1d(np.arange(len(stems)), VAL)
json.dump(TR.tolist(), open(WORK / "train_idx_b.json", "w")); json.dump(CFG, open(WORK / "cfg_b.json", "w"))
print(f"train {len(TR)} / validation {len(VAL)} (same validation images as the first U-Net)")


[0.00 h] DATA + CACHE
707 train images, 1154 annotator records, 180 test images
  cached 1/707 (2s)
  cached 101/707 (39s)
  cached 201/707 (77s)
  cached 301/707 (114s)
  cached 401/707 (152s)
  cached 501/707 (190s)
  cached 601/707 (226s)
  cached 701/707 (263s)
  cached 707/707 (265s)
  cached 1/180 (2s)
  cached 101/180 (36s)
  cached 180/180 (62s)
train 601 / validation 106 (same validation images as the first U-Net)


## 2 · Training on both GPUs (DistributedDataParallel)

In [ ]:
banner("PRETRAINED BACKBONE")
src = B.save_ctx_init(CFG, WORK / "ctx_init_b.pt", log=print)
print("backbone:", CFG["vit"] if CFG["backbone"] == "dinov2" else CFG["cnn"], "| weights:", src or "NONE")
if not src and not DRY:
    raise RuntimeError("No pretrained backbone weights: turn Internet ON, or attach a dataset with the timm weights file.")


def launch(nproc, hours):
    args = ["b_lib.py", "train", "--hours", str(hours), "--work", str(WORK), "--cache", str(TEMP / f"train_{RES}")]
    cmd = ([sys.executable, "-m", "torch.distributed.run", "--standalone", f"--nproc_per_node={nproc}"] + args) if nproc > 1 \
        else [sys.executable] + args
    env = dict(os.environ, PYTHONUNBUFFERED="1", PYTHONPATH=str(WORK), OMP_NUM_THREADS="1" if nproc > 1 else os.environ.get("OMP_NUM_THREADS", ""))
    p = subprocess.Popen(cmd, cwd=str(WORK), env={k: v for k, v in env.items() if v != ""}, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line.rstrip(), flush=True)
    p.wait()
    return p.returncode == 0 and (WORK / "b_model.pt").exists()


banner("TRAINING")
ok = launch(N_PROC, TRAIN_HOURS)
if not ok and N_PROC > 1:
    print("!!! multi-GPU training failed - retrying on a single GPU with the remaining budget")
    ok = launch(1, max(TRAIN_HOURS - (time.time() - T0) / 3600 + 0.3, 0.5))
assert ok, "training failed - see the log above"
TRAIN_INFO = json.load(open(WORK / "b_train.json")); print(TRAIN_INFO)


[0.09 h] PRETRAINED BACKBONE


model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

backbone: vit_base_patch14_reg4_dinov2.lvd142m | weights: timm hub

[0.10 h] TRAINING
2 process(es) | 601 train frames | batch 2/process | 40.4 M trainable | 3.00 h
/kaggle/working/b_lib.py:289: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  t = lambda a: torch.from_numpy(np.ascontiguousarray(a))
/kaggle/working/b_lib.py:289: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warnin

## 3 · Validation: learned query grouping vs connected components, on the first U-Net's validation images

In [ ]:
banner("VALIDATION")
DEV = torch.device("cuda:0" if N_GPU else "cpu")
model = B.build_model(CFG).to(DEV)
model.load_state_dict(torch.load(WORK / "b_model.pt", map_location=DEV, weights_only=True)); model.eval()
t = time.time()
VO = {i: B.infer(model, C["chans"][i], C["gray"][i], C["valid"][i], DEV) for i in VAL}
print(f"{len(VAL)} validation frames in {time.time() - t:.0f}s")
NJ = os.cpu_count() or 1
MIN_AREA = int(40 * (RES / 1024) ** 2)
valid_of = lambda cache, i: np.asarray(cache["valid"][i]).astype(bool)


def gt_res(i):
    labs = [np.asarray(C["labels"][j]) for j in range(C["rec_start"][i], C["rec_start"][i + 1])]
    return {rid: list(S.label_rles(S.relabel(l.astype(np.int32))).values()) for rid, l in zip(data.recs[stems[i]], labs)}


GT_RES = {rid: v for i in VAL for rid, v in gt_res(i).items()}
TAUS = (0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7)


def evaluate(build):
    """build(i) -> (label image, scores). PQ at RES for every score threshold tau."""
    res = dict(zip(VAL, ThreadPoolExecutor(NJ).map(build, VAL)))
    enc = {i: S.label_rles(res[i][0]) for i in VAL}
    out = []
    for tau in TAUS:
        out.append(dict(tau=tau, **S.pq(GT_RES, {stems[i]: [enc[i][k] for k in S.select(res[i][1], tau) if k in enc[i]] for i in VAL})))
    return out


rows = []
for obj in (0.3, 0.5, 0.7):
    for ts in (0.0, 0.3, 0.4, 0.5):
        for r in evaluate(lambda i, obj=obj, ts=ts: B.panoptic(VO[i][0], VO[i][1], VO[i][2], valid_of(C, i), obj, ts, MIN_AREA)):
            rows.append(dict(mode="query", obj_thr=obj, t_sem=ts, **r))


def cc_build(i, t_lo, t_hi):
    sem = VO[i][2]
    lab = S.instances_cc(sem, valid_of(C, i), t_hi, t_lo, MIN_AREA)
    n = int(lab.max())
    return lab, (np.asarray(ndi.mean(sem, lab, np.arange(1, n + 1))) if n else np.zeros(0))


for t_lo, t_hi in [(0.3, 0.5), (0.4, 0.6), (0.5, 0.7)]:
    for r in evaluate(lambda i, a=t_lo, b=t_hi: cc_build(i, a, b)):
        rows.append(dict(mode="cc", obj_thr=t_hi, t_sem=t_lo, **r))
    for obj in (0.3, 0.5, 0.7):                          # hybrid: components + query votes for joining fragments
        for r in evaluate(lambda i, a=t_lo, b=t_hi, obj=obj: B.hybrid(VO[i][0], VO[i][1], VO[i][2], valid_of(C, i), obj, a, b, MIN_AREA)):
            rows.append(dict(mode="hybrid", obj_thr=obj, t_sem=t_lo, t_hi=t_hi, **r))
tab = pd.DataFrame(rows).sort_values("pq", ascending=False).reset_index(drop=True)
print(tab.head(12).round(4).to_string(index=False))
print("\nbest per mode:\n" + tab.groupby("mode").head(1).round(4).to_string(index=False))
BEST = tab.iloc[0].to_dict()


def final_instances(cls_p, mlog, sem, valid, best):
    if best["mode"] == "query":
        lab, sc = B.panoptic(cls_p, mlog, sem, valid, best["obj_thr"], best["t_sem"], MIN_AREA)
        trim = best["t_sem"] if best["t_sem"] > 0 else None    # t_sem 0: raw (dilated) query masks
    elif best["mode"] == "hybrid":
        lab, sc = B.hybrid(cls_p, mlog, sem, valid, best["obj_thr"], best["t_sem"], best["t_hi"], MIN_AREA)
        return lab, sc, best["t_sem"]
    else:
        lab = S.instances_cc(sem, valid, best["obj_thr"], best["t_sem"], MIN_AREA)
        n = int(lab.max())
        sc = np.asarray(ndi.mean(sem, lab, np.arange(1, n + 1))) if n else np.zeros(0)
        trim = best["t_sem"]
    return lab, sc, trim


GT_FULL = data.gt(VAL, S.FULL)
VAL_PQ = {}
for mode in ("query", "hybrid", "cc"):
    bm = tab[tab["mode"] == mode].iloc[0].to_dict()
    preds = {}
    for i in VAL:
        lab, sc, trim = final_instances(*VO[i], valid_of(C, i), bm)
        preds[stems[i]] = S.to_rles(lab, S.select(sc, bm["tau"]), VO[i][2], trim, S.FULL)
    VAL_PQ[mode] = S.pq(GT_FULL, preds)
    print(f"validation PQ @2048, {mode:5s}: {VAL_PQ[mode]}")
BEST = max((tab[tab["mode"] == m].iloc[0].to_dict() for m in VAL_PQ), key=lambda b: VAL_PQ[b["mode"]]["pq"])
print(f"\nchosen for the submission: {BEST}  <- compare its PQ with the first U-Net's validation PQ")

## 4 · Test → submission.csv

In [ ]:
banner("TEST INFERENCE")
test_stems = [p.stem for p in data.test_paths]
t = time.time()
preds = {}
for i, s in enumerate(test_stems):
    cls_p, mlog, sem = B.infer(model, TC["chans"][i], TC["gray"][i], TC["valid"][i], DEV)
    valid = valid_of(TC, i)
    lab, sc, trim = final_instances(cls_p, mlog, sem, valid, BEST)
    rl = S.to_rles(lab, S.select(sc, BEST["tau"]), sem, trim, S.FULL)
    if not rl and sem.max() > 0:                        # nothing found: relax to this image's own probability scale
        t2 = 0.5 * float(sem.max())
        lab = S.instances_cc(sem, valid, 0.8 * float(sem.max()), t2, MIN_AREA)
        if lab.max():
            rl = S.to_rles(lab, [int(np.argmax(np.bincount(lab.ravel())[1:])) + 1], sem, t2, S.FULL)
    preds[s] = rl
sub = S.write_submission(preds, WORK / "submission.csv")
rep = S.validate_submission(sub, test_stems)
print(rep, f"| {time.time() - t:.0f}s")
REPORT = dict(approach="B: DINOv2 (last blocks fine-tuned) + Mask2Former-style decoder + full-res semantic boundaries",
              backbone=CFG["vit"] if CFG["backbone"] == "dinov2" else CFG["cnn"], weights=src, train=TRAIN_INFO,
              validation_pq=VAL_PQ, chosen=BEST, submission=rep, hours=round((time.time() - T0) / 3600, 2))
json.dump(REPORT, open(WORK / "report_b.json", "w"), indent=1, default=str)
print(json.dumps(REPORT, indent=1, default=str))
(WORK / "ctx_init_b.pt").unlink(missing_ok=True)                 # keep the saved version small